# Demo del registry de engine_quant (Fase 3, PLAN.md §5.4/§7.6)

Este notebook es una demo/tutorial del cliente Python (`nanobind`) sobre el registry C++
de modelos, productos y medidas del motor XVA. No repite la validación numérica del
motor (eso vive en los tests de Rust y C++, PLAN.md §5.6) — solo muestra cómo se usa la
API desde Python.

**Antes de ejecutar este notebook**, hay que compilar el proyecto con CMake
(CMake + Corrosion + `cxx` + nanobind, ver `PLAN.md` en la raíz del repo para el proceso
completo). El módulo compilado (`engine.cp3XX-....pyd` en Windows, o `engine*.so` en
Linux/Mac) queda en el directorio de build de CMake, tipicamente `build/clients/python`
relativo a la raíz del repo. Este notebook vive en `clients/python/notebooks/`, tres
niveles por debajo de la raíz, así que añadimos esa ruta relativa al `sys.path` antes
de importar `engine`.

In [ ]:
import sys
from pathlib import Path

# Ruta relativa desde clients/python/notebooks/ hasta el directorio de build de CMake.
# Ajusta esta ruta si tu build vive en otro sitio (ej. build-debug/, o un build fuera del repo).
sys.path.insert(0, "../../../build/clients/python")
sys.path.insert(0, "../src")

import engine
import engine_typed as q

print("módulo engine importado desde:", engine.__file__)

## 1. Arrancar el motor y listar lo que hay registrado

`engine.Engine()` envuelve el registry C++ (`Registries` + `register_builtins`,
PLAN.md §5.4): al construirlo se registran de una vez todos los modelos, productos y
medidas disponibles. `list_models()`/`list_products()`/`list_measures()` permiten
descubrir dinámicamente qué soporta el motor sin mirar el código C++ — exactamente lo
que PLAN.md §5.4 promete: "un vistazo" a todo lo que el motor soporta.

In [ ]:
eng = engine.Engine()

print("modelos disponibles: ", eng.list_models())
print("productos disponibles:", eng.list_products())
print("medidas disponibles: ", eng.list_measures())

## 2. Crear un modelo: Hull-White 1 factor

`q.HullWhite1F(a=..., b=..., sigma=..., r0=...)` (`engine_typed`, `pydantic`) construye un
objeto tipado -- `a` (velocidad de reversión), `b` (nivel de reversión de largo plazo,
constante en este caso base), `sigma` (volatilidad) y `r0` (tipo corto inicial), todos
requeridos. `.to_params()` lo traduce al mismo `Params`/dict que `create_model(nombre, params)`
ya consumía (PLAN.md §5.4) -- sin tocar el core, solo añade validación real antes de llegar
ahí (una clave mal escrita en un dict no se detecta hasta tiempo de ejecución; en `pydantic`
sí, al construir el objeto).

In [ ]:
model_spec = q.HullWhite1F(a=0.1, b=0.03, sigma=0.01, r0=0.02)
model = eng.create_model(model_spec.model_type, model_spec.to_params())
model

## 3. Crear un producto: IRS a la par a 5 años

`q.IRSwap` necesita `notional`, `payment_times` y `accruals` (mismo largo) y un `fixed_rate`
**requerido** -- omitirlo es un `ValidationError` de `pydantic`, no un swap "a la par". Para
eso está el constructor con nombre `IRSwap.par(...)` (propuesta 2 de `PLAN_REAPI.md` §3.2):
el tipo fijo se calcula a mercado (desde la curva de `Market`, ver más abajo).

In [ ]:
trade_spec = q.IRSwap.par(
    notional=1_000_000.0,
    payment_times=[1.0, 2.0, 3.0, 4.0, 5.0],
    accruals=[1.0, 1.0, 1.0, 1.0, 1.0],
)
product = eng.create_product(trade_spec.product_type, trade_spec.to_params())
product

## 4. Market, PricingContext y ExecutionContext (PLAN.md §7.15)

Antes de calcular cualquier medida hace falta un `Market` (curva de descuento observada --
`PV`/`DV01` descuentan por esta curva, `ExpectedExposure`/`PFE95`/`UnilateralCVA` solo usan
`hazard_rate`/`recovery_rate`, ver más abajo, `PLAN_REAPI.md` §6 Fase 4), un `PricingContext`
(fecha de valoración + parámetros de la simulación Monte Carlo: `n_paths`, `n_steps`, `seed`)
y un `ExecutionContext` (cómo ejecutar: `backend` `"cpu"`/`"gpu"`/`"auto"`, `precision`).
Estos tres sustituyen por completo el antiguo rango de parámetros genérico que mezclaba, sin
nombre propio, `monitoring_times`/`n_paths`/`seed` con `hazard_rate`/`recovery_rate`.

In [ ]:
market_spec = q.Market(pillars=[1.0, 2.0], zero_rates=[0.02, 0.02], hazard_rate=0.02, recovery_rate=0.4)
pricing_spec = q.PricingContext(n_paths=5_000, n_steps=208, seed=7)
execution_spec = q.ExecutionContext(backend="auto")

market = engine.MarketSnapshot(**market_spec.to_params())
pricing = engine.PricingContext(pricing_spec.to_params())
execution = engine.ExecutionContext(execution_spec.to_params())

market, pricing, execution

## 5. Perfil de exposición (EE/PFE) vía Monte Carlo

Las medidas `"ExpectedExposure"` y `"PFE95"` de `Engine.price` simulan el tipo corto bajo
Hull-White y revaloran el swap restante en cada trayectoria, devolviendo la exposición
esperada y el PFE al 95% en cada fecha de reseteo del propio swap (auto-derivadas del
producto, ya no un `monitoring_times` que haya que pasar a mano). `Engine.price` las calcula
juntas en una única simulación al pedirlas en el mismo lote.

In [ ]:
exposure = eng.price(product, ["ExpectedExposure", "PFE95"], model, market, pricing, execution)

print("times:   ", exposure["ExpectedExposure"].times)
print("EE:      ", exposure["ExpectedExposure"].primary)
print("PFE(95%):", exposure["PFE95"].primary)

### Graficar EE/PFE

Requiere `matplotlib` instalado en el entorno de Jupyter (no es una dependencia del
motor en sí, solo de este notebook).

In [ ]:
import matplotlib.pyplot as plt

plt.plot(exposure["ExpectedExposure"].times, exposure["ExpectedExposure"].primary, marker="o", label="EE")
plt.plot(exposure["PFE95"].times, exposure["PFE95"].primary, marker="o", label="PFE (95%)")
plt.xlabel("tiempo (años)")
plt.ylabel("exposición")
plt.title("Perfil de exposición IRS bajo Hull-White 1F")
plt.legend()
plt.grid(True)
plt.show()

## 6. CVA unilateral

La medida `"UnilateralCVA"` de `Engine.price` reutiliza `"ExpectedExposure"` por composición
(no vuelve a correr la simulación Monte Carlo aparte, PLAN.md §7.6) y aplica la
`hazard_rate`/`recovery_rate` de `market` para llegar al CVA agregado. El resultado escalar
viene en `result.scalar` (con `result.has_scalar == True`; las medidas de perfil como
`ExpectedExposure` no rellenan este campo).

In [ ]:
cva = eng.price(product, ["UnilateralCVA"], model, market, pricing, execution)

print("CVA unilateral:", cva["UnilateralCVA"].scalar)

## 7. Calibración: ajustar un modelo a un `MarketSnapshot` (PLAN.md §7.14/§7.18)

En vez de elegir los parámetros del modelo a mano, un calibrador los ajusta a una curva
de mercado (`MarketSnapshot`, aquí fabricada con `synthetic_from_hull_white*` -- útil
para probar/demostrar calibración sin depender de datos reales). `list_calibrators()`/
`create_calibrator(nombre)`/`Calibrator.calibrate(market, estimacion_inicial)` son tan
genéricos por nombre como `create_model`/`create_product`: el motor ya tiene **dos**
calibradores, uno por modelo, y no son el mismo código con los nombres cambiados --
`HullWhite1F` calibra `a` (velocidad de reversión) y `b` (nivel de largo plazo,
cualquier signo); `HullWhite2F`/G2++ calibra `a` y `b`, las velocidades de reversión de
**ambos** factores latentes (las dos deben ser positivas). En los dos casos, `sigma`
(y en G2++ también `eta`/`rho`) se toman como datos de entrada: no están bien
identificados contra únicamente una curva de descuento (harían falta instrumentos de
volatilidad, swaptions/caps, fuera de alcance hoy).

In [ ]:
print("calibradores disponibles:", eng.list_calibrators())

# --- HullWhite1F: calibra (a, b) ---
market_1f = engine.MarketSnapshot.synthetic_from_hull_white(
    a=0.15, b=0.025, sigma=0.008, r0=0.02,
    pillars=[0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0],
)
calibrator_1f = eng.create_calibrator("HullWhite1F")
# Estimación inicial deliberadamente lejos de los parámetros "verdaderos" -- tipada igual
# que cualquier otro HullWhite1F, solo que aquí sirve de punto de partida, no de resultado.
initial_guess_1f = q.HullWhite1F(a=0.3, b=0.01, sigma=0.008, r0=0.02)
result_1f = calibrator_1f.calibrate(market_1f, initial_guess_1f.to_params())
print("HullWhite1F:", result_1f)
print("  optimal_params:", result_1f.optimal_params)

# El resultado alimenta directamente create_model -- cierra el círculo
# Mercado -> calibrar -> Modelo calibrado. q.HullWhite1F(**...) valida el resultado antes
# de volver a pasar por create_model.
calibrated_model_1f = eng.create_model("HullWhite1F", q.HullWhite1F(**result_1f.optimal_params).to_params())
calibrated_model_1f

In [ ]:
# --- HullWhite2F/G2++: calibra (a, b), las dos velocidades de reversión ---
market_2f = engine.MarketSnapshot.synthetic_from_hull_white_2f(
    a=0.15, b=0.25, sigma=0.008, eta=0.01, rho=-0.6, r0=0.02,
    pillars=[0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0],
)
calibrator_2f = eng.create_calibrator("HullWhite2F")
initial_guess_2f = q.HullWhite2F(a=0.4, b=0.05, sigma=0.008, eta=0.01, rho=-0.6, r0=0.02)
result_2f = calibrator_2f.calibrate(market_2f, initial_guess_2f.to_params())
print("HullWhite2F:", result_2f)
print("  optimal_params:", result_2f.optimal_params)

calibrated_model_2f = eng.create_model("HullWhite2F", q.HullWhite2F(**result_2f.optimal_params).to_params())
calibrated_model_2f

## 8. Lotes: `price_batch`, `price_many`, `price_grid` (PLAN.md §7.17/§7.19)

Tres niveles, cada uno construido sobre el anterior:

- **`price_batch`** -- lote *homogéneo*: N trades del mismo tipo/calendario, vectorizado
  sin bucle escalar (el tipo corto se simula una única vez para todo el lote). Cada trade
  de `"IRSwap"` debe traer `fixed_rate` explícito -- el lote no soporta `IRSwap.par(...)`.
- **`price_many`** -- lote *heterogéneo*: admite trades de tipos/calendarios distintos, los
  agrupa internamente (por `(tipo, calendario)`) y llama a `price_batch` por grupo -- nunca
  falla por heterogeneidad.
- **`price_grid`** -- la explosión de combinaciones **Trades × Models × Markets**: por cada
  par (modelo, mercado) llama a `price_many` sobre todos los trades. `PricingContext`/
  `ExecutionContext` son compartidos, no forman parte de la rejilla.

Las tres devuelven una fila por (trade[, modelo, mercado]) con su índice explícito -- nunca
una lista anidada -- mismo diseño en las cinco capas (C++/C ABI/Python/Excel).

In [ ]:
# --- price_batch: 3 swaps del mismo calendario, cada uno con su propio notional/fixed_rate ---
schedule = {"payment_times": [1.0, 2.0, 3.0, 4.0, 5.0], "accruals": [1.0] * 5}
trade_a_spec = q.IRSwap(notional=1_000_000.0, fixed_rate=0.02, **schedule)
trade_b_spec = q.IRSwap(notional=2_500_000.0, fixed_rate=0.015, **schedule)
trade_c_spec = q.IRSwap(notional=500_000.0, fixed_rate=0.025, **schedule)

trade_a = eng.create_product(trade_a_spec.product_type, trade_a_spec.to_params())
trade_b = eng.create_product(trade_b_spec.product_type, trade_b_spec.to_params())
trade_c = eng.create_product(trade_c_spec.product_type, trade_c_spec.to_params())

batch = eng.price_batch([trade_a, trade_b, trade_c], ["PV", "UnilateralCVA"], model, market, pricing, execution)
for row in batch:
    print(f"trade_index={row.trade_index}  PV={row.measures['PV'].scalar:.2f}  "
          f"CVA={row.measures['UnilateralCVA'].scalar:.2f}")

In [ ]:
# --- price_many: trades de calendarios distintos, agrupados internamente ---
trade_3y_spec = q.IRSwap(notional=2_000_000.0, fixed_rate=0.018,
                          payment_times=[1.0, 2.0, 3.0], accruals=[1.0] * 3)
trade_3y = eng.create_product(trade_3y_spec.product_type, trade_3y_spec.to_params())

# Intercalados a propósito: 5y, 3y, 5y -- dos grupos de calendario, no en bloques contiguos.
many = eng.price_many([trade_a, trade_3y, trade_b], ["PV"], model, market, pricing, execution)
for row in many:
    print(f"trade_index={row.trade_index}  PV={row.measures['PV'].scalar:.2f}")

In [ ]:
# --- price_grid: Trades x Models x Markets ---
model_2f_spec = q.HullWhite2F(a=0.1, b=0.2, sigma=0.01, eta=0.012, rho=-0.7, r0=0.03)
market_stressed_spec = q.Market(pillars=[1.0, 2.0], zero_rates=[0.05, 0.05], hazard_rate=0.05, recovery_rate=0.3)

model_2f = eng.create_model(model_2f_spec.model_type, model_2f_spec.to_params())
market_stressed = engine.MarketSnapshot(**market_stressed_spec.to_params())

grid = eng.price_grid([trade_a, trade_b], ["PV", "UnilateralCVA"],
                       [model, model_2f], [market, market_stressed], pricing, execution)
print(f"{len(grid)} celdas (2 trades x 2 modelos x 2 mercados)")
for cell in grid:
    print(f"  trade={cell.trade_index} model={cell.model_index} market={cell.market_index}  "
          f"PV={cell.measures['PV'].scalar:.2f}")

## 9. Extensibilidad del registry

El punto central de PLAN.md §5.4 es que **añadir un modelo/producto/medida nuevo no
requiere tocar los clientes**: basta con implementar la interfaz correspondiente en C++
(`IModel`/`IProduct`/`IMeasure`) y añadir una línea de registro en
`engine::register_builtins` (`cpp/engine/src/bootstrap.cpp`) -- y, si la medida nueva
debe ser invocable desde `ENGINE.PRICE`, una entrada más en la tabla de
`cpp/engine/src/price.cpp` (PLAN.md §7.15). En cuanto ese tipo nuevo existe ahí, aparece
automáticamente aquí, sin recompilar ni tocar este notebook:

In [ ]:
# Vuelve a listar lo disponible: si alguien añade, por ejemplo, un modelo Black-Scholes o
# una medida FVA en cpp/engine/src/bootstrap.cpp, aparecería aquí sin cambiar este notebook.
print("modelos disponibles: ", eng.list_models())
print("productos disponibles:", eng.list_products())
print("medidas disponibles: ", eng.list_measures())